# Point-BERT dVAE Tokenizer + Masked Pretraining on NuScenes
## Full Implementation with 3D Plotly Visualizations

Based on: *Point-BERT: Pre-training 3D Point Cloud Transformers with Masked Point Modeling* (CVPR 2022)

**Stage 1 â€” dVAE Tokenizer:**
- FPS + kNN patch partition
- PointNet-style patch encoder
- DGCNN for token logits
- Gumbel-Softmax straight-through VQ
- Folding-based decoder (coarse + fine)
- Chamfer Distance + KL divergence training

**Stage 2 â€” Masked Point Modeling (MPM):**
- Transformer backbone with positional embedding
- Masked patches â†’ predict frozen dVAE tokens
- Cross-entropy loss on masked positions


In [3]:
!pip -q install nuscenes-devkit pyquaternion
!pip -q install "numpy>=1.22.0,<2.0.0"
!pip -q install plotly
print('Dependencies installed')

Dependencies installed


In [2]:

# DOWNLOAD FULL NUSCENES MINI DATA

import os

 

# Download metadata
!wget -nc https://www.nuscenes.org/data/v1.0-mini.tgz
!tar -xzf v1.0-mini.tgz

# Download samples (images + lidar)
!wget -nc https://www.nuscenes.org/data/v1.0-mini-samples.tgz
!tar -xzf v1.0-mini-samples.tgz

# Download sweeps (required for lidar)
!wget -nc https://www.nuscenes.org/data/v1.0-mini-sweeps.tgz
!tar -xzf v1.0-mini-sweeps.tgz

print("NuScenes mini fully downloaded.")


'wget' is not recognized as an internal or external command,
operable program or batch file.
tar: Error opening archive: Failed to open 'v1.0-mini.tgz'
'wget' is not recognized as an internal or external command,
operable program or batch file.
tar: Error opening archive: Failed to open 'v1.0-mini-samples.tgz'
'wget' is not recognized as an internal or external command,
operable program or batch file.


NuScenes mini fully downloaded.


tar: Error opening archive: Failed to open 'v1.0-mini-sweeps.tgz'


In [ ]:
# DOWNLOAD LIDARSEG LABELS (per-point semantic labels)
import os
if not os.path.exists(os.path.join(NUSCENES_DATAROOT, 'lidarseg')):
    !wget -nc https://www.nuscenes.org/data/nuScenes-lidarseg-mini-v1.0.tar.bz2
    !tar -xjf nuScenes-lidarseg-mini-v1.0.tar.bz2
    print('LiDAR segmentation labels downloaded.')
else:
    print('LiDAR segmentation labels already present.')


In [ ]:
import os, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import LidarPointCloud
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
!pip install nuscenes-devkit pyquaternion
!apt-get install -y unzip

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

Device: cpu


'apt-get' is not recognized as an internal or external command,
operable program or batch file.


## Configuration

In [4]:
# â”€â”€ Data â”€â”€
NUSCENES_DATAROOT = '/content'
NUSCENES_VERSION  = 'v1.0-mini'

# â”€â”€ Point cloud grouping â”€â”€
N_POINTS   = 1024     # points sampled per frame
NUM_GROUP  = 64       # number of patches (groups)
GROUP_SIZE = 32       # points per patch

# â”€â”€ dVAE architecture â”€â”€
NUM_TOKENS   = 512   # codebook size
TOKENS_DIMS  = 256    # codebook embedding dim
ENCODER_DIMS = 256    # patch encoder output dim
DECODER_DIMS = 256    # decoder input dim

# â”€â”€ dVAE training â”€â”€
DVAE_EPOCHS   = 50
DVAE_LR       = 1e-3
DVAE_BATCH    = 8
TEMP_START    = 1.0   # Gumbel temperature start
TEMP_END      = 0.5   # Gumbel temperature end
KL_WEIGHT     = 0.1   # KL divergence weight

# â”€â”€ Point-BERT (Stage 2) â”€â”€
TRANS_DIM   = 384     # transformer hidden dim
DEPTH       = 6       # transformer blocks
NUM_HEADS   = 6       # attention heads
MLP_RATIO   = 4.0
DROP_RATE   = 0.0
DROP_PATH   = 0.1
MASK_RATIO  = 0.6     # fraction of patches to mask
BERT_EPOCHS = 50
BERT_LR     = 5e-4
BERT_BATCH  = 8


## FPS + kNN Patch Creation

In [5]:
@torch.no_grad()
def farthest_point_sampling(xyz, n_samples):
    N = xyz.shape[0]
    centroids = torch.zeros(n_samples, dtype=torch.long, device=xyz.device)
    distance = torch.full((N,), 1e10, device=xyz.device)
    farthest = torch.randint(0, N, (1,), device=xyz.device).item()
    for i in range(n_samples):
        centroids[i] = farthest
        centroid = xyz[farthest].view(1, 3)
        dist = torch.sum((xyz - centroid) ** 2, dim=1)
        distance = torch.minimum(distance, dist)
        farthest = torch.argmax(distance).item()
    return centroids

@torch.no_grad()
def knn_indices(xyz, centers, k):
    d = torch.cdist(centers, xyz)
    return torch.topk(d, k=k, largest=False).indices

@torch.no_grad()
def make_patches(points_xyz, num_group, group_size):
    fps_idx = farthest_point_sampling(points_xyz, num_group)
    centers = points_xyz[fps_idx]
    nn_idx = knn_indices(points_xyz, centers, group_size)
    patches = points_xyz[nn_idx]
    patches = patches - centers[:, None, :]
    return patches, centers

def fps_batch(xyz, n_samples):
    """Batch FPS: xyz (B, N, 3) -> centers (B, n_samples, 3), idx (B, n_samples)"""
    B, N, _ = xyz.shape
    centroids = torch.zeros(B, n_samples, dtype=torch.long, device=xyz.device)
    distance = torch.full((B, N), 1e10, device=xyz.device)
    farthest = torch.randint(0, N, (B,), device=xyz.device)
    for i in range(n_samples):
        centroids[:, i] = farthest
        centroid = xyz[torch.arange(B, device=xyz.device), farthest].unsqueeze(1)
        dist = torch.sum((xyz - centroid) ** 2, dim=-1)
        distance = torch.minimum(distance, dist)
        farthest = torch.argmax(distance, dim=-1)
    return centroids

def make_patches_batch(xyz, num_group, group_size):
    """Batch version: xyz (B, N, 3) -> patches (B, G, K, 3), centers (B, G, 3)"""
    B, N, _ = xyz.shape
    fps_idx = fps_batch(xyz, num_group)  # (B, G)
    centers = torch.gather(xyz, 1, fps_idx.unsqueeze(-1).expand(-1, -1, 3))  # (B, G, 3)
    dists = torch.cdist(centers, xyz)  # (B, G, N)
    _, nn_idx = torch.topk(dists, k=group_size, largest=False)  # (B, G, K)
    nn_idx_exp = nn_idx.unsqueeze(-1).expand(-1, -1, -1, 3)  # (B, G, K, 3)
    patches = torch.gather(xyz.unsqueeze(1).expand(-1, num_group, -1, -1), 2, nn_idx_exp)
    patches = patches - centers.unsqueeze(2)  # center-normalize
    return patches, centers

## NuScenes LiDAR Dataset

In [6]:
class NuScenesLidarDataset(Dataset):
    """Returns raw point clouds (N_POINTS, 3) for batch processing."""
    def __init__(self, dataroot, version):
        self.nusc = NuScenes(version=version, dataroot=dataroot, verbose=False)
        self.tokens = [s['data']['LIDAR_TOP'] for s in self.nusc.sample]
        # Check for lidarseg availability
        self.has_lidarseg = hasattr(self.nusc, 'lidarseg') and len(self.nusc.lidarseg) > 0
        if not self.has_lidarseg:
            # Try loading lidarseg
            lidarseg_path = os.path.join(dataroot, 'lidarseg')
            self.has_lidarseg = os.path.exists(lidarseg_path)
        print(f'LiDAR Segmentation available: {self.has_lidarseg}')

    def __len__(self): return len(self.tokens)

    def __getitem__(self, idx):
        sd = self.nusc.get('sample_data', self.tokens[idx])
        lidar_path = os.path.join(self.nusc.dataroot, sd['filename'])
        pc = LidarPointCloud.from_file(lidar_path)
        pts = pc.points[:3].T.astype(np.float32)
        pts = pts[np.isfinite(pts).all(axis=1)]
        choice = np.random.choice(pts.shape[0], N_POINTS, replace=False)
        return torch.from_numpy(pts[choice])  # (N_POINTS, 3)


## Chamfer Distance (Pure PyTorch)

In [ ]:
def chamfer_distance_l1(preds, gts):
    """L1 Chamfer Distance: preds (B, N, 3), gts (B, M, 3) -> scalar"""
    dist = torch.cdist(preds, gts, p=2)  # (B, N, M)
    min1, _ = dist.min(dim=2)  # (B, N)
    min2, _ = dist.min(dim=1)  # (B, M)
    return (min1.mean() + min2.mean()) / 2

def chamfer_distance_l2(preds, gts):
    """L2 Chamfer Distance (squared): preds (B, N, 3), gts (B, M, 3) -> scalar"""
    dist = torch.cdist(preds, gts, p=2) ** 2  # (B, N, M)
    min1, _ = dist.min(dim=2)
    min2, _ = dist.min(dim=1)
    return (min1.mean() + min2.mean()) / 2

## DGCNN (Dynamic Graph CNN)

In [8]:
def knn_graph(x, k=4):
    """x: (B, C, N) -> idx: (B, N, k)"""
    x_t = x.permute(0, 2, 1)  # (B, N, C)
    dist = torch.cdist(x_t, x_t)  # (B, N, N)
    _, idx = dist.topk(k, dim=-1, largest=False)  # (B, N, k)
    return idx

def get_graph_feature(coor, x, k=4):
    """
    coor: (B, 3, N) coordinates for kNN
    x:    (B, C, N) features
    Returns: (B, 2C, N, k)
    """
    B, C, N = x.shape
    # kNN on coordinates
    coor_t = coor.permute(0, 2, 1)  # (B, N, 3)
    dist = torch.cdist(coor_t, coor_t)  # (B, N, N)
    _, idx = dist.topk(k, dim=-1, largest=False)  # (B, N, k)
    # gather neighbor features
    idx_base = torch.arange(0, B, device=x.device).view(-1, 1, 1) * N
    idx_flat = (idx + idx_base).view(-1)
    x_t = x.permute(0, 2, 1).contiguous().view(B * N, -1)  # (BN, C)
    neighbors = x_t[idx_flat].view(B, N, k, C).permute(0, 3, 1, 2)  # (B, C, N, k)
    x_exp = x.unsqueeze(-1).expand(-1, -1, -1, k)  # (B, C, N, k)
    return torch.cat([neighbors - x_exp, x_exp], dim=1)  # (B, 2C, N, k)

class DGCNN(nn.Module):
    """DGCNN from Point-BERT paper. Maps (B, G, C_in) + coords (B, G, 3) -> (B, G, C_out)"""
    def __init__(self, encoder_channel, output_channel):
        super().__init__()
        self.input_trans = nn.Conv1d(encoder_channel, 128, 1)
        self.layer1 = nn.Sequential(
            nn.Conv2d(256, 256, 1, bias=False), nn.GroupNorm(4, 256), nn.LeakyReLU(0.2))
        self.layer2 = nn.Sequential(
            nn.Conv2d(512, 512, 1, bias=False), nn.GroupNorm(4, 512), nn.LeakyReLU(0.2))
        self.layer3 = nn.Sequential(
            nn.Conv2d(1024, 512, 1, bias=False), nn.GroupNorm(4, 512), nn.LeakyReLU(0.2))
        self.layer4 = nn.Sequential(
            nn.Conv2d(1024, 1024, 1, bias=False), nn.GroupNorm(4, 1024), nn.LeakyReLU(0.2))
        self.layer5 = nn.Sequential(
            nn.Conv1d(2304, output_channel, 1, bias=False),
            nn.GroupNorm(4, output_channel), nn.LeakyReLU(0.2))

    def forward(self, f, coor):
        """f: (B, G, C), coor: (B, G, 3) -> (B, G, C_out)"""
        coor = coor.transpose(1, 2).contiguous()  # B 3 G
        f = f.transpose(1, 2).contiguous()         # B C G
        f = self.input_trans(f)                     # B 128 G
        feat_list = []
        f = get_graph_feature(coor, f, k=4)         # B 256 G k
        f = self.layer1(f).max(dim=-1)[0]; feat_list.append(f)
        f = get_graph_feature(coor, f, k=4)
        f = self.layer2(f).max(dim=-1)[0]; feat_list.append(f)
        f = get_graph_feature(coor, f, k=4)
        f = self.layer3(f).max(dim=-1)[0]; feat_list.append(f)
        f = get_graph_feature(coor, f, k=4)
        f = self.layer4(f).max(dim=-1)[0]; feat_list.append(f)
        f = torch.cat(feat_list, dim=1)             # B 2304 G
        f = self.layer5(f)                          # B C_out G
        return f.transpose(1, 2)                    # B G C_out

## PointNet-style Patch Encoder

In [9]:
class PatchEncoder(nn.Module):
    """Encodes each patch (group of points) into a feature vector.
    Input: (B, G, K, 3) -> Output: (B, G, encoder_dims)"""
    def __init__(self, encoder_channel=ENCODER_DIMS):
        super().__init__()
        self.encoder_channel = encoder_channel
        self.first_conv = nn.Sequential(
            nn.Conv1d(3, 128, 1), nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.Conv1d(128, 256, 1))
        self.second_conv = nn.Sequential(
            nn.Conv1d(512, 512, 1), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Conv1d(512, encoder_channel, 1))

    def forward(self, patches):
        """patches: (B, G, K, 3) -> (B, G, C)"""
        B, G, K, _ = patches.shape
        x = patches.reshape(B * G, K, 3)
        x = self.first_conv(x.transpose(2, 1))          # (BG, 256, K)
        x_global = x.max(dim=2, keepdim=True)[0]        # (BG, 256, 1)
        x = torch.cat([x_global.expand(-1, -1, K), x], dim=1)  # (BG, 512, K)
        x = self.second_conv(x)                         # (BG, C, K)
        x = x.max(dim=2)[0]                             # (BG, C)
        return x.reshape(B, G, self.encoder_channel)

## Folding-based Decoder

In [10]:
class FoldingDecoder(nn.Module):
    """Decodes codebook embeddings back to point patches.
    Produces coarse (G, num_coarse, 3) + fine (G, group_size, 3) points."""
    def __init__(self, encoder_channel=DECODER_DIMS, num_fine=GROUP_SIZE):
        super().__init__()
        self.num_fine = num_fine
        self.grid_size = 2
        self.num_coarse = num_fine // 4
        assert num_fine % 4 == 0

        self.mlp = nn.Sequential(
            nn.Linear(encoder_channel, 1024), nn.ReLU(inplace=True),
            nn.Linear(1024, 1024), nn.ReLU(inplace=True),
            nn.Linear(1024, 3 * self.num_coarse))
        self.final_conv = nn.Sequential(
            nn.Conv1d(encoder_channel + 3 + 2, 512, 1),
            nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Conv1d(512, 512, 1),
            nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Conv1d(512, 3, 1))

        # 2D folding grid seed
        a = torch.linspace(-0.05, 0.05, self.grid_size).view(1, self.grid_size).expand(self.grid_size, -1).reshape(1, -1)
        b = torch.linspace(-0.05, 0.05, self.grid_size).view(self.grid_size, 1).expand(-1, self.grid_size).reshape(1, -1)
        self.register_buffer('folding_seed', torch.cat([a, b], dim=0).view(1, 2, self.grid_size ** 2))

    def forward(self, feature_global):
        """feature_global: (B, G, C) -> coarse (B, G, M, 3), fine (B, G, K, 3)"""
        B, G, C = feature_global.shape
        x = feature_global.reshape(B * G, C)
        coarse = self.mlp(x).reshape(B * G, self.num_coarse, 3)  # (BG, M, 3)

        # expand coarse points with grid
        point_feat = coarse.unsqueeze(2).expand(-1, -1, self.grid_size ** 2, -1)
        point_feat = point_feat.reshape(B * G, self.num_fine, 3).transpose(2, 1)  # (BG, 3, N)

        seed = self.folding_seed.unsqueeze(2).expand(B * G, -1, self.num_coarse, -1)
        seed = seed.reshape(B * G, 2, self.num_fine)  # (BG, 2, N)

        global_feat = x.unsqueeze(2).expand(-1, -1, self.num_fine)  # (BG, C, N)
        feat = torch.cat([global_feat, seed, point_feat], dim=1)  # (BG, C+5, N)

        center = coarse.unsqueeze(2).expand(-1, -1, self.grid_size ** 2, -1)
        center = center.reshape(B * G, self.num_fine, 3).transpose(2, 1)  # (BG, 3, N)

        fine = self.final_conv(feat) + center  # (BG, 3, N)
        fine = fine.reshape(B, G, 3, self.num_fine).transpose(-1, -2)  # (B, G, K, 3)
        coarse = coarse.reshape(B, G, self.num_coarse, 3)
        return coarse, fine

## Discrete VAE (dVAE) â€” Full Model

In [11]:
class DiscreteVAE(nn.Module):
    """
    Point-BERT dVAE Tokenizer.
    Forward: patches -> Encoder -> DGCNN1 -> Gumbel-Softmax -> codebook lookup
             -> DGCNN2 -> FoldingDecoder -> reconstructed patches
    """
    def __init__(self):
        super().__init__()
        self.encoder = PatchEncoder(encoder_channel=ENCODER_DIMS)
        self.dgcnn_1 = DGCNN(encoder_channel=ENCODER_DIMS, output_channel=NUM_TOKENS)
        self.codebook = nn.Parameter(torch.randn(NUM_TOKENS, TOKENS_DIMS))
        self.dgcnn_2 = DGCNN(encoder_channel=TOKENS_DIMS, output_channel=DECODER_DIMS)
        self.decoder = FoldingDecoder(encoder_channel=DECODER_DIMS, num_fine=GROUP_SIZE)

    def get_tokens(self, patches, centers):
        """Get discrete token IDs for patches (no grad). patches: (B,G,K,3), centers: (B,G,3)"""
        with torch.no_grad():
            logits = self.encoder(patches)
            logits = self.dgcnn_1(logits, centers)
            return logits.argmax(dim=-1)  # (B, G)

    def forward(self, patches, centers, temperature=1.0, hard=False):
        """patches: (B,G,K,3), centers: (B,G,3)"""
        # Encode patches
        encoded = self.encoder(patches)               # (B, G, ENCODER_DIMS)
        logits = self.dgcnn_1(encoded, centers)       # (B, G, NUM_TOKENS)

        # Gumbel-Softmax (straight-through)
        soft_one_hot = F.gumbel_softmax(logits, tau=temperature, dim=2, hard=hard)  # (B, G, NUM_TOKENS)

        # Codebook lookup
        sampled = torch.einsum('b g n, n c -> b g c', soft_one_hot, self.codebook)  # (B, G, TOKENS_DIMS)

        # Decode
        feature = self.dgcnn_2(sampled, centers)      # (B, G, DECODER_DIMS)
        coarse, fine = self.decoder(feature)           # coarse: (B,G,M,3), fine: (B,G,K,3)

        return coarse, fine, patches, logits  # patches = ground truth

    def compute_loss(self, coarse, fine, gt_patches, logits):
        """Compute Chamfer + KL loss."""
        B, G, _, _ = coarse.shape
        # Flatten groups for Chamfer
        coarse_flat = coarse.reshape(B * G, -1, 3)
        fine_flat = fine.reshape(B * G, -1, 3)
        gt_flat = gt_patches.reshape(B * G, -1, 3)

        loss_coarse = chamfer_distance_l1(coarse_flat, gt_flat)
        loss_fine = chamfer_distance_l1(fine_flat, gt_flat)
        loss_recon = loss_coarse + loss_fine

        # KL divergence: encourage uniform codebook usage
        softmax = F.softmax(logits, dim=-1)           # (B, G, NUM_TOKENS)
        mean_softmax = softmax.mean(dim=1)            # (B, NUM_TOKENS)
        log_qy = torch.log(mean_softmax + 1e-10)
        log_uniform = torch.log(torch.tensor(1.0 / NUM_TOKENS, device=logits.device))
        loss_kl = F.kl_div(log_qy, log_uniform.expand_as(log_qy),
                           reduction='batchmean', log_target=True)

        return loss_recon, loss_kl

print('dVAE architecture defined.')
# Quick shape test
_dvae = DiscreteVAE()
_p = torch.randn(2, NUM_GROUP, GROUP_SIZE, 3)
_c = torch.randn(2, NUM_GROUP, 3)
_co, _fi, _gt, _lo = _dvae(_p, _c)
print(f'coarse: {_co.shape}, fine: {_fi.shape}, logits: {_lo.shape}')
del _dvae, _p, _c, _co, _fi, _gt, _lo

dVAE architecture defined.


coarse: torch.Size([2, 64, 8, 3]), fine: torch.Size([2, 64, 32, 3]), logits: torch.Size([2, 64, 8192])


## Stage 1: dVAE Training Loop

In [12]:
def train_dvae(model, dataset, epochs=DVAE_EPOCHS, lr=DVAE_LR, batch_size=DVAE_BATCH):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    # Train/val split
    n = len(dataset)
    n_train = max(1, int(0.8 * n))
    train_ds, val_ds = torch.utils.data.random_split(dataset, [n_train, n - n_train])
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

    history = {'train_loss': [], 'val_loss': [], 'recon': [], 'kl': []}

    for epoch in range(1, epochs + 1):
        # Temperature annealing
        temp = TEMP_START + (TEMP_END - TEMP_START) * (epoch - 1) / max(1, epochs - 1)

        # â”€â”€ Train â”€â”€
        model.train()
        total_loss, total_recon, total_kl, count = 0, 0, 0, 0
        for pts in train_dl:
            pts = pts.to(DEVICE)  # (B, N_POINTS, 3)
            patches, centers = make_patches_batch(pts, NUM_GROUP, GROUP_SIZE)
            coarse, fine, gt, logits = model(patches, centers, temperature=temp, hard=False)
            loss_recon, loss_kl = model.compute_loss(coarse, fine, gt, logits)
            loss = loss_recon + KL_WEIGHT * loss_kl

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            bs = pts.size(0)
            total_loss += loss.item() * bs
            total_recon += loss_recon.item() * bs
            total_kl += loss_kl.item() * bs
            count += bs

        scheduler.step()
        train_loss = total_loss / count
        history['train_loss'].append(train_loss)
        history['recon'].append(total_recon / count)
        history['kl'].append(total_kl / count)

        # â”€â”€ Val â”€â”€
        model.eval()
        val_total, val_count = 0, 0
        with torch.no_grad():
            for pts in val_dl:
                pts = pts.to(DEVICE)
                patches, centers = make_patches_batch(pts, NUM_GROUP, GROUP_SIZE)
                coarse, fine, gt, logits = model(patches, centers, temperature=temp, hard=True)
                loss_r, loss_k = model.compute_loss(coarse, fine, gt, logits)
                val_total += (loss_r + KL_WEIGHT * loss_k).item() * pts.size(0)
                val_count += pts.size(0)
        val_loss = val_total / max(1, val_count)
        history['val_loss'].append(val_loss)

        if epoch % 5 == 0 or epoch == 1:
            print(f'Epoch {epoch:3d}/{epochs} | Temp {temp:.3f} | '
                  f'Train {train_loss:.4f} (recon {history["recon"][-1]:.4f}, kl {history["kl"][-1]:.4f}) | '
                  f'Val {val_loss:.4f} | LR {scheduler.get_last_lr()[0]:.2e}')

    print('\nâœ“ dVAE training complete.')
    return history

### Run dVAE Training

In [13]:
# Load dataset
dataset = NuScenesLidarDataset(NUSCENES_DATAROOT, NUSCENES_VERSION)
print(f'Dataset: {len(dataset)} LiDAR frames')

# Create and train dVAE
dvae = DiscreteVAE()
n_params = sum(p.numel() for p in dvae.parameters()) / 1e6
print(f'dVAE parameters: {n_params:.2f}M')

dvae_history = train_dvae(dvae, dataset)

AssertionError: Database version not found: /content\v1.0-mini

### dVAE Loss Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(dvae_history['train_loss'], label='Train', marker='o', markersize=3)
ax1.plot(dvae_history['val_loss'], label='Val', marker='o', markersize=3)
ax1.set_title('dVAE Total Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(dvae_history['recon'], label='Chamfer (recon)', marker='s', markersize=3)
ax2.plot(dvae_history['kl'], label='KL divergence', marker='^', markersize=3)
ax2.set_title('Loss Components'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

## dVAE Visualizations

In [ ]:
def plot_pc(points, title='Point Cloud', size=2, color=None):
    if isinstance(points, torch.Tensor): points = points.detach().cpu().numpy()
    fig = go.Figure(data=[go.Scatter3d(
        x=points[:,0], y=points[:,1], z=points[:,2], mode='markers',
        marker=dict(size=size, color=color if color is not None else points[:,2], colorscale='Viridis')
    )])
    fig.update_layout(scene_aspectmode='data', title=title)
    fig.show()

def compare_patch(original, reconstructed):
    if isinstance(original, torch.Tensor): original = original.detach().cpu().numpy()
    if isinstance(reconstructed, torch.Tensor): reconstructed = reconstructed.detach().cpu().numpy()
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=original[:,0], y=original[:,1], z=original[:,2],
                               mode='markers', name='Original', marker=dict(size=3, color='blue')))
    fig.add_trace(go.Scatter3d(x=reconstructed[:,0], y=reconstructed[:,1], z=reconstructed[:,2],
                               mode='markers', name='Reconstructed', marker=dict(size=3, color='red')))
    fig.update_layout(scene_aspectmode='data', title='Patch Reconstruction')
    fig.show()

### Tokenization + Reconstruction Demo

In [ ]:
dvae.eval()
# Get one sample
sample_pts = dataset[0].unsqueeze(0).to(DEVICE)  # (1, N, 3)
patches, centers = make_patches_batch(sample_pts, NUM_GROUP, GROUP_SIZE)

with torch.no_grad():
    coarse, fine, gt, logits = dvae(patches, centers, temperature=TEMP_END, hard=True)
    token_ids = logits.argmax(dim=-1).squeeze(0)  # (G,)

# 1) Raw point cloud
raw_pts = (patches.squeeze(0) + centers.squeeze(0).unsqueeze(1)).reshape(-1, 3)
plot_pc(raw_pts, title='Original LiDAR Patches', size=2)

# 2) Token-colored patches
colors = np.repeat(token_ids.cpu().numpy(), GROUP_SIZE)
plot_pc(raw_pts, title='Token-Colored Patches', size=3, color=colors)

# 3) Reconstruction
rec_pts = (fine.squeeze(0) + centers.squeeze(0).unsqueeze(1)).reshape(-1, 3)
plot_pc(rec_pts, title='dVAE Reconstruction', size=2)

# 4) Compare a single patch
compare_patch(gt[0, 0], fine[0, 0])

# 5) Token distribution
all_tokens = []
dvae.eval()
with torch.no_grad():
    for i in range(min(len(dataset), 20)):
        pts = dataset[i].unsqueeze(0).to(DEVICE)
        pa, ce = make_patches_batch(pts, NUM_GROUP, GROUP_SIZE)
        tids = dvae.get_tokens(pa, ce).cpu()
        all_tokens.append(tids)
all_tokens = torch.cat(all_tokens, dim=0).reshape(-1).numpy()
plt.figure(figsize=(12, 3))
plt.hist(all_tokens, bins=min(100, NUM_TOKENS), edgecolor='black', alpha=0.7)
plt.title(f'Token Distribution ({len(all_tokens)} tokens, {len(np.unique(all_tokens))} unique)')
plt.xlabel('Token ID'); plt.ylabel('Count'); plt.tight_layout(); plt.show()

---
## Stage 2: Masked Point Modeling (Point-BERT)

Freeze the dVAE and train a Transformer backbone to predict
masked patch tokens via cross-entropy loss.

In [ ]:
class TransformerBlock(nn.Module):
    """Pre-norm transformer block with stochastic depth."""
    def __init__(self, dim, num_heads, mlp_ratio=4.0, drop=0.0, drop_path=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=drop, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, dim), nn.Dropout(drop))
        self.drop_path = drop_path

    def forward(self, x):
        # Stochastic depth
        if self.training and self.drop_path > 0 and random.random() < self.drop_path:
            return x
        h = self.norm1(x)
        h, _ = self.attn(h, h, h)
        x = x + h
        x = x + self.mlp(self.norm2(x))
        return x

class PointBERT(nn.Module):
    """
    Point-BERT: Masked Point Modeling with frozen dVAE tokens as targets.
    """
    def __init__(self, frozen_dvae):
        super().__init__()
        self.dvae = frozen_dvae
        for p in self.dvae.parameters():
            p.requires_grad = False
        self.dvae.eval()

        # Patch encoder (shared architecture, trained from scratch for transformer)
        self.encoder = PatchEncoder(encoder_channel=ENCODER_DIMS)
        self.reduce_dim = nn.Linear(ENCODER_DIMS, TRANS_DIM)

        # Learnable tokens
        self.cls_token = nn.Parameter(torch.randn(1, 1, TRANS_DIM) * 0.02)
        self.mask_token = nn.Parameter(torch.randn(1, 1, TRANS_DIM) * 0.02)
        self.cls_pos = nn.Parameter(torch.randn(1, 1, TRANS_DIM) * 0.02)

        # Positional embedding from 3D coordinates
        self.pos_embed = nn.Sequential(
            nn.Linear(3, 128), nn.GELU(), nn.Linear(128, TRANS_DIM))

        # Transformer blocks
        dpr = [x.item() for x in torch.linspace(0, DROP_PATH, DEPTH)]
        self.blocks = nn.ModuleList([
            TransformerBlock(TRANS_DIM, NUM_HEADS, MLP_RATIO, DROP_RATE, dpr[i])
            for i in range(DEPTH)])
        self.norm = nn.LayerNorm(TRANS_DIM)

        # Token prediction head
        self.lm_head = nn.Linear(TRANS_DIM, NUM_TOKENS)

    def _mask_center(self, center):
        """Spatially contiguous masking. center: (B, G, 3) -> mask: (B, G) bool"""
        B, G, _ = center.shape
        masks = []
        for b in range(B):
            idx = random.randint(0, G - 1)
            dists = torch.norm(center[b] - center[b, idx:idx+1], dim=-1)  # (G,)
            sorted_idx = torch.argsort(dists)
            n_mask = int(MASK_RATIO * G)
            mask = torch.zeros(G, dtype=torch.bool, device=center.device)
            mask[sorted_idx[:n_mask]] = True
            masks.append(mask)
        return torch.stack(masks)  # (B, G)

    def forward(self, patches, centers):
        """
        patches: (B,G,K,3), centers: (B,G,3)
        Returns: logits (B,G,NUM_TOKENS), dvae_labels (B,G), mask (B,G)
        """
        B, G = patches.shape[:2]

        # Get dVAE ground truth tokens
        with torch.no_grad():
            dvae_labels = self.dvae.get_tokens(patches, centers)  # (B, G)

        # Generate mask
        bool_mask = self._mask_center(centers)  # (B, G)

        # Encode patches
        group_tokens = self.encoder(patches)  # (B, G, ENCODER_DIMS)
        group_tokens = self.reduce_dim(group_tokens)  # (B, G, TRANS_DIM)

        # Apply mask: replace masked tokens with mask_token
        w = bool_mask.unsqueeze(-1).float()  # (B, G, 1)
        masked_tokens = group_tokens * (1 - w) + self.mask_token.expand(B, G, -1) * w

        # Prepend CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)  # (B, 1, D)
        x = torch.cat([cls_tokens, masked_tokens], dim=1)  # (B, G+1, D)

        # Positional embedding
        pos = self.pos_embed(centers)  # (B, G, D)
        cls_pos = self.cls_pos.expand(B, -1, -1)
        pos = torch.cat([cls_pos, pos], dim=1)  # (B, G+1, D)

        # Transformer
        for block in self.blocks:
            x = block(x + pos)
        x = self.norm(x)

        # Predict tokens (skip CLS)
        logits = self.lm_head(x[:, 1:])  # (B, G, NUM_TOKENS)

        return logits, dvae_labels, bool_mask

print('Point-BERT architecture defined.')

### Point-BERT Training Loop

In [ ]:
def compute_cosine_similarity(logits, labels, codebook):
    """Cosine similarity between predicted (soft) and GT token embeddings."""
    pred_probs = F.softmax(logits, dim=-1)                        # (N, V)
    pred_embed = torch.matmul(pred_probs, codebook)               # (N, D)
    gt_embed   = codebook[labels]                                  # (N, D)
    return F.cosine_similarity(pred_embed, gt_embed, dim=-1).mean()

def compute_topk_accuracy(logits, labels, k=5):
    """Check if GT token is in top-k predictions."""
    topk = logits.topk(k, dim=-1).indices
    return (topk == labels.unsqueeze(-1)).any(-1).float().mean()

def compute_confidence(logits):
    """Mean max-softmax confidence."""
    return F.softmax(logits, dim=-1).max(-1).values.mean()

def train_point_bert(model, dataset, epochs=BERT_EPOCHS, lr=BERT_LR, batch_size=BERT_BATCH):
    model = model.to(DEVICE)
    train_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(train_params, lr=lr, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    n = len(dataset); n_train = max(1, int(0.8 * n))
    train_ds, val_ds = torch.utils.data.random_split(dataset, [n_train, n - n_train])
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    val_dl   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    codebook = model.dvae.codebook.detach()  # frozen codebook

    history = {k: [] for k in [
        'train_loss','val_loss','train_acc','val_acc',
        'train_cos_sim','val_cos_sim',
        'train_top5','val_top5','train_top10','val_top10',
        'train_conf','val_conf']}

    print(f'\n========== STAGE 2 \u2013 MASKED POINT MODELING ==========')
    print(f'Metrics: Loss | Exact-Acc | Cosine-Sim | Top-5 | Top-10 | Confidence\n')

    for epoch in range(1, epochs + 1):
        model.train(); model.dvae.eval()
        s = {k: 0.0 for k in ['l','a','c','t5','t10','cf','n']}
        for pts in train_dl:
            pts = pts.to(DEVICE)
            patches, centers = make_patches_batch(pts, NUM_GROUP, GROUP_SIZE)
            logits, labels, mask = model(patches, centers)
            ml, la = logits[mask], labels[mask]
            loss = F.cross_entropy(ml, la)
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(train_params, 1.0); optimizer.step()
            bs = pts.size(0); s['n'] += bs
            with torch.no_grad():
                s['l']  += loss.item() * bs
                s['a']  += (ml.argmax(-1) == la).float().mean().item() * bs
                s['c']  += compute_cosine_similarity(ml, la, codebook).item() * bs
                s['t5'] += compute_topk_accuracy(ml, la, k=5).item() * bs
                s['t10']+= compute_topk_accuracy(ml, la, k=10).item() * bs
                s['cf'] += compute_confidence(ml).item() * bs
        scheduler.step()
        for key, skey in [('train_loss','l'),('train_acc','a'),('train_cos_sim','c'),
                          ('train_top5','t5'),('train_top10','t10'),('train_conf','cf')]:
            history[key].append(s[skey] / s['n'])

        # ── Validation ──
        model.eval()
        v = {k: 0.0 for k in ['l','a','c','t5','t10','cf','n']}
        with torch.no_grad():
            for pts in val_dl:
                pts = pts.to(DEVICE)
                patches, centers = make_patches_batch(pts, NUM_GROUP, GROUP_SIZE)
                logits, labels, mask = model(patches, centers)
                ml, la = logits[mask], labels[mask]
                loss = F.cross_entropy(ml, la)
                bs = pts.size(0); v['n'] += bs
                v['l']  += loss.item() * bs
                v['a']  += (ml.argmax(-1) == la).float().mean().item() * bs
                v['c']  += compute_cosine_similarity(ml, la, codebook).item() * bs
                v['t5'] += compute_topk_accuracy(ml, la, k=5).item() * bs
                v['t10']+= compute_topk_accuracy(ml, la, k=10).item() * bs
                v['cf'] += compute_confidence(ml).item() * bs
        vn = max(1, v['n'])
        for key, skey in [('val_loss','l'),('val_acc','a'),('val_cos_sim','c'),
                          ('val_top5','t5'),('val_top10','t10'),('val_conf','cf')]:
            history[key].append(v[skey] / vn)

        if epoch % 5 == 0 or epoch == 1:
            h = history
            print(f'Ep {epoch:3d}/{epochs} | '
                  f'Loss {h["train_loss"][-1]:.4f}/{h["val_loss"][-1]:.4f} | '
                  f'Acc {h["train_acc"][-1]:.4f}/{h["val_acc"][-1]:.4f} | '
                  f'CosSim {h["train_cos_sim"][-1]:.4f}/{h["val_cos_sim"][-1]:.4f} | '
                  f'Top5 {h["train_top5"][-1]:.4f} | '
                  f'Top10 {h["train_top10"][-1]:.4f} | '
                  f'Conf {h["train_conf"][-1]:.4f}')

    print('\n\u2714 Point-BERT pretraining complete.')
    return history


### Run Point-BERT Training

In [ ]:
point_bert = PointBERT(frozen_dvae=dvae)
n_params = sum(p.numel() for p in point_bert.parameters() if p.requires_grad) / 1e6
print(f'Point-BERT trainable parameters: {n_params:.2f}M')

bert_history = train_point_bert(point_bert, dataset)

### Point-BERT Training Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Row 1: Loss, Exact Accuracy, Cosine Similarity
axes[0,0].plot(bert_history['train_loss'], label='Train'); axes[0,0].plot(bert_history['val_loss'], label='Val')
axes[0,0].set_title('Cross-Entropy Loss'); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(bert_history['train_acc'], label='Train'); axes[0,1].plot(bert_history['val_acc'], label='Val')
axes[0,1].set_title('Exact Token Accuracy'); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

axes[0,2].plot(bert_history['train_cos_sim'], label='Train'); axes[0,2].plot(bert_history['val_cos_sim'], label='Val')
axes[0,2].set_title('Cosine Similarity (Embedding)'); axes[0,2].set_ylim(-0.1, 1.05); axes[0,2].legend(); axes[0,2].grid(alpha=0.3)

# Row 2: Top-5, Top-10, Confidence
axes[1,0].plot(bert_history['train_top5'], label='Train'); axes[1,0].plot(bert_history['val_top5'], label='Val')
axes[1,0].set_title('Top-5 Accuracy'); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

axes[1,1].plot(bert_history['train_top10'], label='Train'); axes[1,1].plot(bert_history['val_top10'], label='Val')
axes[1,1].set_title('Top-10 Accuracy'); axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

axes[1,2].plot(bert_history['train_conf'], label='Train'); axes[1,2].plot(bert_history['val_conf'], label='Val')
axes[1,2].set_title('Prediction Confidence'); axes[1,2].legend(); axes[1,2].grid(alpha=0.3)

for ax in axes.flat: ax.set_xlabel('Epoch')
plt.suptitle('Stage 2 \u2013 Masked Point Modeling Metrics', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


### Masked Point Modeling Visualization

In [ ]:
point_bert.eval()
codebook = point_bert.dvae.codebook.detach()
sample_pts = dataset[0].unsqueeze(0).to(DEVICE)
patches, centers = make_patches_batch(sample_pts, NUM_GROUP, GROUP_SIZE)

with torch.no_grad():
    logits, dvae_labels, mask = point_bert(patches, centers)
    pred_tokens = logits.argmax(dim=-1).squeeze(0)  # (G,)
    gt_tokens = dvae_labels.squeeze(0)
    mask_1d = mask.squeeze(0)
    logits_1d = logits.squeeze(0)  # (G, NUM_TOKENS)

patches_np = patches.squeeze(0).cpu().numpy()
centers_np = centers.squeeze(0).cpu().numpy()

# ── Per-patch cosine similarity for masked patches ──
masked_logits = logits_1d[mask_1d]      # (M, V)
masked_labels = gt_tokens[mask_1d]      # (M,)
pred_probs = F.softmax(masked_logits, dim=-1)
pred_embed = torch.matmul(pred_probs, codebook)
gt_embed = codebook[masked_labels]
per_patch_cos = F.cosine_similarity(pred_embed, gt_embed, dim=-1).cpu().numpy()

# ── Per-patch confidence ──
per_patch_conf = pred_probs.max(-1).values.cpu().numpy()

# ── Summary statistics ──
masked_pred = pred_tokens[mask_1d].cpu().numpy()
masked_gt = gt_tokens[mask_1d].cpu().numpy()
exact_acc = (masked_pred == masked_gt).mean()
top5_acc = compute_topk_accuracy(masked_logits, masked_labels, k=5).item()
top10_acc = compute_topk_accuracy(masked_logits, masked_labels, k=10).item()
mean_cos = per_patch_cos.mean()
mean_conf = per_patch_conf.mean()

print('=' * 60)
print('  STAGE 2 \u2013 MASKED PREDICTION SIMILARITY REPORT')
print('=' * 60)
print(f'  Exact Token Accuracy : {exact_acc:.4f}')
print(f'  Cosine Similarity    : {mean_cos:.4f}')
print(f'  Top-5 Accuracy       : {top5_acc:.4f}')
print(f'  Top-10 Accuracy      : {top10_acc:.4f}')
print(f'  Mean Confidence      : {mean_conf:.4f}')
print('=' * 60)

# ── Plot 1: Visible vs Masked, colored by cosine similarity ──
fig = go.Figure()
mask_counter = 0
for i in range(NUM_GROUP):
    pts = patches_np[i] + centers_np[i]
    if mask_1d[i]:
        sim = per_patch_cos[mask_counter]
        mask_counter += 1
        fig.add_trace(go.Scatter3d(
            x=pts[:,0], y=pts[:,1], z=pts[:,2], mode='markers',
            marker=dict(size=3, color=sim, colorscale='RdYlGn', cmin=0, cmax=1,
                        colorbar=dict(title='Cosine Sim') if mask_counter==1 else None),
            name=f'Masked #{mask_counter} (sim={sim:.3f})', showlegend=False))
    else:
        fig.add_trace(go.Scatter3d(
            x=pts[:,0], y=pts[:,1], z=pts[:,2], mode='markers',
            marker=dict(size=2, color='lightblue', opacity=0.3), showlegend=False))
fig.update_layout(scene_aspectmode='data',
    title=f'Masked Patches Colored by Cosine Similarity (mean={mean_cos:.4f})')
fig.show()

# ── Plot 2: Per-patch similarity histogram ──
fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.hist(per_patch_cos, bins=20, edgecolor='black', alpha=0.7, color='green')
ax1.axvline(mean_cos, color='red', linestyle='--', label=f'Mean={mean_cos:.4f}')
ax1.set_title('Per-Patch Cosine Similarity Distribution')
ax1.set_xlabel('Cosine Similarity'); ax1.legend()

ax2.hist(per_patch_conf, bins=20, edgecolor='black', alpha=0.7, color='orange')
ax2.axvline(mean_conf, color='red', linestyle='--', label=f'Mean={mean_conf:.4f}')
ax2.set_title('Per-Patch Prediction Confidence')
ax2.set_xlabel('Confidence'); ax2.legend()
plt.tight_layout(); plt.show()

# ── Print sample predictions with similarity ──
print(f'\nSample masked patches (first 15):')
print(f'{"Patch":>6} {"Pred":>8} {"GT":>8} {"Match":>6} {"CosSim":>8} {"Conf":>8}')
for j in range(min(15, len(masked_pred))):
    match = '\u2714' if masked_pred[j] == masked_gt[j] else '\u2718'
    print(f'{j:6d} {masked_pred[j]:8d} {masked_gt[j]:8d} {match:>6} {per_patch_cos[j]:8.4f} {per_patch_conf[j]:8.4f}')


## Save Pretrained Models

In [ ]:
# Save dVAE
torch.save(dvae.state_dict(), 'dvae_nuscenes.pth')
print('dVAE saved â†’ dvae_nuscenes.pth')

# Save Point-BERT (encoder + transformer only, no frozen dVAE)
bert_state = {
    'encoder': point_bert.encoder.state_dict(),
    'reduce_dim': point_bert.reduce_dim.state_dict(),
    'blocks': point_bert.blocks.state_dict(),
    'norm': point_bert.norm.state_dict(),
    'lm_head': point_bert.lm_head.state_dict(),
    'pos_embed': point_bert.pos_embed.state_dict(),
    'cls_token': point_bert.cls_token,
    'mask_token': point_bert.mask_token,
    'cls_pos': point_bert.cls_pos,
}
torch.save(bert_state, 'pointbert_nuscenes.pth')
print('Point-BERT saved â†’ pointbert_nuscenes.pth')
print('\nâœ“ All done!')

---
## Stage 3: Downstream Validation with LiDAR Segmentation Labels

Using **per-point semantic segmentation labels** from NuScenes to assign
real-world categories to every patch. Each patch gets the **majority class**
of its 32 LiDAR points.


In [ ]:
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import plotly.express as px
from collections import Counter

# NuScenes lidarseg class names (idx -> name)
LIDARSEG_NAMES = {
    0: 'noise', 1: 'animal', 2: 'human.pedestrian.adult',
    3: 'human.pedestrian.child', 4: 'human.pedestrian.construction_worker',
    5: 'human.pedestrian.personal_mobility', 6: 'human.pedestrian.police_officer',
    7: 'human.pedestrian.stroller', 8: 'human.pedestrian.wheelchair',
    9: 'movable_object.barrier', 10: 'movable_object.debris',
    11: 'movable_object.pushable_pullable', 12: 'movable_object.trafficcone',
    13: 'static_object.bicycle_rack', 14: 'vehicle.bicycle',
    15: 'vehicle.bus.bendy', 16: 'vehicle.bus.rigid', 17: 'vehicle.car',
    18: 'vehicle.construction', 19: 'vehicle.emergency.ambulance',
    20: 'vehicle.emergency.police', 21: 'vehicle.motorcycle',
    22: 'vehicle.trailer', 23: 'vehicle.truck',
    24: 'flat.driveable_surface', 25: 'flat.other', 26: 'flat.sidewalk',
    27: 'flat.terrain', 28: 'static.manmade', 29: 'static.other',
    30: 'static.vegetation', 31: 'vehicle.ego',
}

# Simplified category mapping
def simplify_lidarseg(class_idx):
    name = LIDARSEG_NAMES.get(class_idx, 'unknown')
    if 'car' in name: return 'Car'
    if 'truck' in name: return 'Truck'
    if 'bus' in name: return 'Bus'
    if 'motorcycle' in name: return 'Motorcycle'
    if 'bicycle' in name and 'rack' not in name: return 'Bicycle'
    if 'pedestrian' in name: return 'Pedestrian'
    if 'construction' in name: return 'Construction'
    if 'trailer' in name: return 'Trailer'
    if 'barrier' in name: return 'Barrier'
    if 'trafficcone' in name: return 'Traffic Cone'
    if 'driveable' in name: return 'Road'
    if 'sidewalk' in name: return 'Sidewalk'
    if 'terrain' in name: return 'Terrain'
    if 'vegetation' in name: return 'Vegetation'
    if 'manmade' in name: return 'Building'
    if name == 'noise': return 'Noise'
    return 'Other'

print('Validation libraries loaded.')


### Step 1: Load Per-Point Semantic Labels

For each LiDAR frame, load the semantic label for every point,
then assign each patch the **majority class** of its 32 points.


In [ ]:
def get_lidarseg_labels(nusc, sample_data_token, point_indices):
    """
    Load per-point lidarseg labels for a LiDAR frame.
    point_indices: which points were selected (from random sampling)
    Returns: array of class indices for selected points
    """
    sd = nusc.get('sample_data', sample_data_token)
    lidarseg_path = os.path.join(nusc.dataroot,
        f'lidarseg/v1.0-mini/{sample_data_token}_lidarseg.bin')
    
    if os.path.exists(lidarseg_path):
        labels = np.fromfile(lidarseg_path, dtype=np.uint8)
        return labels[point_indices]
    return None

def get_patch_labels(point_labels, nn_indices):
    """
    Assign each patch the majority class of its points.
    point_labels: (N,) per-point class indices
    nn_indices: (G, K) indices into points for each patch
    Returns: list of simplified category names for each patch
    """
    patch_labels = []
    for g in range(nn_indices.shape[0]):
        patch_point_labels = point_labels[nn_indices[g]]
        majority_class = Counter(patch_point_labels.tolist()).most_common(1)[0][0]
        patch_labels.append(simplify_lidarseg(majority_class))
    return patch_labels

print('Label loading functions defined.')


### Step 2: Extract Embeddings + Semantic Labels for All Frames


In [ ]:
def extract_all_with_lidarseg(model, nusc, max_samples=None):
    """
    Extract Point-BERT embeddings and lidarseg labels for all frames.
    """
    model.eval()
    all_feats, all_centers, all_tokens, all_labels = [], [], [], []
    n = min(len(nusc.sample), max_samples) if max_samples else len(nusc.sample)
    skipped = 0

    with torch.no_grad():
        for i in range(n):
            sample = nusc.sample[i]
            sd_token = sample['data']['LIDAR_TOP']
            sd = nusc.get('sample_data', sd_token)

            # Load point cloud
            lidar_path = os.path.join(nusc.dataroot, sd['filename'])
            pc = LidarPointCloud.from_file(lidar_path)
            all_pts = pc.points[:3].T.astype(np.float32)
            valid_mask = np.isfinite(all_pts).all(axis=1)
            all_pts = all_pts[valid_mask]

            # Sample points (same as dataset)
            choice = np.random.choice(all_pts.shape[0], N_POINTS, replace=False)
            pts_np = all_pts[choice]
            pts = torch.from_numpy(pts_np).unsqueeze(0).to(DEVICE)  # (1,N,3)

            # Make patches
            patches, centers = make_patches_batch(pts, NUM_GROUP, GROUP_SIZE)

            # Get kNN indices to know which points are in each patch
            dists = torch.cdist(centers, pts)  # (1, G, N)
            _, nn_idx = torch.topk(dists.squeeze(0), k=GROUP_SIZE, largest=False)  # (G, K)
            nn_idx_np = nn_idx.cpu().numpy()

            # Load lidarseg labels
            # Map valid_mask indices: choice -> original indices
            valid_indices = np.where(valid_mask)[0]
            original_indices = valid_indices[choice]

            lidarseg_path = os.path.join(nusc.dataroot,
                f'lidarseg/v1.0-mini/{sd_token}_lidarseg.bin')
            if not os.path.exists(lidarseg_path):
                skipped += 1
                continue

            all_point_labels = np.fromfile(lidarseg_path, dtype=np.uint8)
            # Get labels for our sampled points
            selected_labels = all_point_labels[original_indices]
            # Get majority label for each patch
            patch_cats = get_patch_labels(selected_labels, nn_idx_np)

            # Get dVAE tokens
            token_ids = model.dvae.get_tokens(patches, centers).squeeze(0).cpu().numpy()

            # Forward through transformer (no masking)
            group_tokens = model.encoder(patches)
            group_tokens = model.reduce_dim(group_tokens)
            cls = model.cls_token.expand(1, -1, -1)
            x = torch.cat([cls, group_tokens], dim=1)
            pos = model.pos_embed(centers)
            cls_pos = model.cls_pos.expand(1, -1, -1)
            pos = torch.cat([cls_pos, pos], dim=1)
            for block in model.blocks:
                x = block(x + pos)
            x = model.norm(x)

            all_feats.append(x[:, 1:].squeeze(0).cpu().numpy())
            all_centers.append(centers.squeeze(0).cpu().numpy())
            all_tokens.append(token_ids)
            all_labels.extend(patch_cats)

            if (i + 1) % 50 == 0:
                print(f'  Processed {i+1}/{n} frames...')

    patch_feats = np.concatenate(all_feats, axis=0)
    patch_centers = np.concatenate(all_centers, axis=0)
    token_ids = np.concatenate(all_tokens, axis=0)
    labels = np.array(all_labels)

    print(f'\nExtracted {patch_feats.shape[0]} patches with lidarseg labels')
    if skipped > 0:
        print(f'  ({skipped} frames skipped — no lidarseg file)')

    unique, counts = np.unique(labels, return_counts=True)
    print('\nCategory distribution:')
    for cat, cnt in sorted(zip(unique, counts), key=lambda x: -x[1]):
        print(f'  {cat:20s}: {cnt:5d} patches ({100*cnt/len(labels):.1f}%)')

    return patch_feats, patch_centers, token_ids, labels

nusc = dataset.nusc
patch_feats, patch_centers, token_ids, patch_labels = \
    extract_all_with_lidarseg(point_bert, nusc)


### Step 3: t-SNE — Do Real Semantic Categories Cluster?


In [ ]:
MAX_PATCHES = 5000
n_patches = patch_feats.shape[0]
idx = np.random.choice(n_patches, min(MAX_PATCHES, n_patches), replace=False)

sub_feats = patch_feats[idx]
sub_labels = patch_labels[idx]
sub_tokens = token_ids[idx]

print(f'Running t-SNE on {len(idx)} patches...')
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
tsne_2d = tsne.fit_transform(sub_feats)
print('t-SNE complete.')

color_map = {
    'Car': '#e74c3c', 'Truck': '#e67e22', 'Bus': '#f39c12',
    'Pedestrian': '#9b59b6', 'Motorcycle': '#1abc9c', 'Bicycle': '#2ecc71',
    'Barrier': '#00bcd4', 'Traffic Cone': '#e91e63',
    'Construction': '#795548', 'Trailer': '#607d8b',
    'Road': '#4caf50', 'Sidewalk': '#8bc34a', 'Terrain': '#cddc39',
    'Vegetation': '#009688', 'Building': '#3f51b5',
    'Noise': '#9e9e9e', 'Other': '#bdbdbd',
}

fig = px.scatter(x=tsne_2d[:, 0], y=tsne_2d[:, 1], color=sub_labels,
                 color_discrete_map=color_map,
                 title='t-SNE of Point-BERT Embeddings \u2014 Colored by Semantic Category',
                 labels={'x': 't-SNE 1', 'y': 't-SNE 2', 'color': 'Category'},
                 width=950, height=700, opacity=0.7)
fig.update_traces(marker_size=5)
fig.show()


### Step 4: Token \u2192 Semantic Category Mapping

Which real-world categories does each codebook token represent?


In [ ]:
# Build token -> category distribution
token_to_cats = {}
for tok, label in zip(token_ids, patch_labels):
    if tok not in token_to_cats:
        token_to_cats[tok] = []
    token_to_cats[tok].append(label)

token_counts = Counter(token_ids)
top_tokens = token_counts.most_common(25)

print('=' * 80)
print('  TOKEN \u2192 SEMANTIC CATEGORY MAPPING (Top 25 tokens)')
print('=' * 80)
print(f'{"Token":>6} {"Count":>6}  {"Primary Category":>15} {"Purity":>7}  {"Distribution"}')
print('-' * 80)

token_primary_cat = {}
purities = []
for tok, count in top_tokens:
    cat_counter = Counter(token_to_cats[tok])
    total = sum(cat_counter.values())
    primary = cat_counter.most_common(1)[0]
    purity = primary[1] / total
    token_primary_cat[tok] = primary[0]
    purities.append(purity)
    top3 = ', '.join([f'{c}:{n}' for c, n in cat_counter.most_common(3)])
    print(f'{tok:6d} {count:6d}  {primary[0]:>15s} {purity:6.0%}   [{top3}]')

# Overall stats
all_purities = []
for tok, cats in token_to_cats.items():
    if len(cats) >= 3:
        cc = Counter(cats)
        all_purities.append(cc.most_common(1)[0][1] / len(cats))
mean_purity = np.mean(all_purities) if all_purities else 0

print(f'\nOverall token purity: {mean_purity:.2%}')
print(f'Unique tokens used: {len(token_to_cats)}/{NUM_TOKENS}')


### Step 5: Masked Prediction with Semantic Categories & Cosine Similarity

For each masked patch, show the predicted token, GT token,
what category each represents, and cosine similarity.


In [ ]:
# Run masked prediction across MULTIPLE frames for better stats
point_bert.eval()
codebook = point_bert.dvae.codebook.detach()
nusc = dataset.nusc

all_gt_cats, all_pred_cats = [], []
all_cos_sims = []
all_correct = []
cat_cos_sims = {}
cat_correct = Counter()
cat_total = Counter()
detail_rows = []  # for display

def get_token_category(tok):
    if tok in token_primary_cat:
        return token_primary_cat[tok]
    if tok in token_to_cats and len(token_to_cats[tok]) > 0:
        return Counter(token_to_cats[tok]).most_common(1)[0][0]
    return '?'

n_frames_to_eval = min(len(nusc.sample), 50)  # eval on up to 50 frames
print(f'Running masked predictions on {n_frames_to_eval} frames...\n')

with torch.no_grad():
    for i in range(n_frames_to_eval):
        sd_token = nusc.sample[i]['data']['LIDAR_TOP']
        sd = nusc.get('sample_data', sd_token)
        lidar_path = os.path.join(nusc.dataroot, sd['filename'])
        pc = LidarPointCloud.from_file(lidar_path)
        all_pts = pc.points[:3].T.astype(np.float32)
        valid_mask = np.isfinite(all_pts).all(axis=1)
        all_pts = all_pts[valid_mask]
        choice = np.random.choice(all_pts.shape[0], N_POINTS, replace=False)
        pts = torch.from_numpy(all_pts[choice]).unsqueeze(0).to(DEVICE)

        patches, centers = make_patches_batch(pts, NUM_GROUP, GROUP_SIZE)
        logits, dvae_labels, mask = point_bert(patches, centers)

        # kNN indices for label lookup
        dists_knn = torch.cdist(centers, pts)
        _, nn_idx = torch.topk(dists_knn.squeeze(0), k=GROUP_SIZE, largest=False)
        nn_idx_np = nn_idx.cpu().numpy()

        # Lidarseg labels
        lidarseg_path = os.path.join(nusc.dataroot,
            f'lidarseg/v1.0-mini/{sd_token}_lidarseg.bin')
        if not os.path.exists(lidarseg_path):
            continue
        all_point_labels = np.fromfile(lidarseg_path, dtype=np.uint8)
        valid_indices = np.where(valid_mask)[0]
        selected_labels = all_point_labels[valid_indices[choice]]
        frame_cats = get_patch_labels(selected_labels, nn_idx_np)

        # Cosine sim for masked patches
        pred_tokens = logits.argmax(-1).squeeze(0)
        gt_tokens = dvae_labels.squeeze(0)
        mask_1d = mask.squeeze(0)
        ml = logits.squeeze(0)[mask_1d]
        la = gt_tokens[mask_1d]

        pred_probs = F.softmax(ml, dim=-1)
        pred_embed = torch.matmul(pred_probs, codebook)
        gt_embed = codebook[la]
        cos_sims = F.cosine_similarity(pred_embed, gt_embed, dim=-1).cpu().numpy()
        conf = pred_probs.max(-1).values.cpu().numpy()

        m_pred = pred_tokens[mask_1d].cpu().numpy()
        m_gt = gt_tokens[mask_1d].cpu().numpy()
        masked_cats = [frame_cats[j] for j in range(NUM_GROUP) if mask_1d[j]]

        for j in range(len(m_pred)):
            gt_cat = masked_cats[j]
            pred_cat = get_token_category(m_pred[j])
            correct = m_pred[j] == m_gt[j]
            cs = cos_sims[j]

            cat_total[gt_cat] += 1
            if correct:
                cat_correct[gt_cat] += 1
            if gt_cat not in cat_cos_sims:
                cat_cos_sims[gt_cat] = []
            cat_cos_sims[gt_cat].append(cs)
            all_cos_sims.append(cs)
            all_correct.append(correct)

            if len(detail_rows) < 30:
                detail_rows.append((
                    len(detail_rows), m_pred[j], m_gt[j],
                    '\u2714' if correct else '\u2718',
                    cs, conf[j], gt_cat, pred_cat
                ))

# Print detailed table
print('=' * 95)
print('  MASKED PREDICTION WITH SEMANTIC CATEGORIES (sample of 30 patches)')
print('=' * 95)
print(f'{"Patch":>6} {"Pred":>6} {"GT":>6} {"Match":>6} {"CosSim":>8} '
      f'{"Conf":>6} {"GT Category":>15} {"Pred Category":>15}')
print('-' * 95)
for row in detail_rows:
    print(f'{row[0]:6d} {row[1]:6d} {row[2]:6d} {row[3]:>6} '
          f'{row[4]:8.4f} {row[5]:6.4f} {row[6]:>15} {row[7]:>15}')

# Per-category summary
print(f'\n\n' + '=' * 70)
print('  PER-CATEGORY PREDICTION QUALITY (aggregated over all frames)')
print('=' * 70)
print(f'{"Category":>15} {"Patches":>8} {"Accuracy":>10} {"Avg CosSim":>12} {"Std":>8}')
print('-' * 70)
for cat in sorted(cat_total.keys(), key=lambda c: -cat_total[c]):
    acc = cat_correct[cat] / cat_total[cat] if cat_total[cat] > 0 else 0
    avg_cos = np.mean(cat_cos_sims[cat]) if cat in cat_cos_sims else 0
    std_cos = np.std(cat_cos_sims[cat]) if cat in cat_cos_sims else 0
    print(f'{cat:>15} {cat_total[cat]:8d} {acc:10.2%} {avg_cos:12.4f} {std_cos:8.4f}')

print(f'\n  Overall: {np.mean(all_correct):.2%} accuracy, '
      f'{np.mean(all_cos_sims):.4f} avg cosine similarity')


### Step 6: Category-Level Visualization


In [ ]:
# Bar charts: accuracy and cosine similarity per category
cats_sorted = [c for c in sorted(cat_total.keys(), key=lambda c: -cat_total[c])
               if cat_total[c] >= 5]  # only categories with enough samples

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, max(5, len(cats_sorted)*0.4)))

cat_means = [np.mean(cat_cos_sims[c]) for c in cats_sorted]
cat_stds = [np.std(cat_cos_sims[c]) for c in cats_sorted]
ax1.barh(cats_sorted, cat_means, xerr=cat_stds, color='#3498db', alpha=0.8)
ax1.set_xlabel('Cosine Similarity'); ax1.set_title('Avg Cosine Similarity by Category')
ax1.set_xlim(0, 1); ax1.invert_yaxis()

cat_accs = [cat_correct[c]/cat_total[c] for c in cats_sorted]
ax2.barh(cats_sorted, cat_accs, color='#2ecc71', alpha=0.8)
ax2.set_xlabel('Exact Accuracy'); ax2.set_title('Token Prediction Accuracy by Category')
ax2.set_xlim(0, 1); ax2.invert_yaxis()

plt.suptitle('Point-BERT Performance by Semantic Category', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
n_unique = len(np.unique(token_ids))
overall_acc = np.mean(all_correct)
overall_cos = np.mean(all_cos_sims)

print('\n' + '=' * 65)
print('  DOWNSTREAM VALIDATION SUMMARY')
print('=' * 65)
print(f'  Dataset:                {len(dataset)} LiDAR frames')
print(f'  Total patches:          {len(patch_labels)}')
print(f'  Semantic categories:    {len(np.unique(patch_labels))}')
print(f'  Codebook size:          {NUM_TOKENS}')
print(f'  Codebook utilization:   {n_unique}/{NUM_TOKENS} '
      f'({100*n_unique/NUM_TOKENS:.1f}%)')
print(f'  Token purity:           {mean_purity:.2%}')
print(f'  Overall exact accuracy: {overall_acc:.2%}')
print(f'  Overall cosine sim:     {overall_cos:.4f}')
print('=' * 65)
